# Results

Loads saved metrics/history from `results/` and produces the figures used in
the top-level README. Run this after `scripts/run_all_experiments.sh` finishes.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path('../results')

def load_history(kind, run_name):
    with open(RESULTS / kind / run_name / 'history.json') as f:
        return json.load(f)

def load_eval(run_name, split):
    tag = f"{run_name}__{split.replace('/', '_')}"
    with open(RESULTS / 'tables' / f'{tag}.json') as f:
        return json.load(f)

## 1-1: Architecture comparison (baseline / wide / deep)

In [ ]:
arch_runs = ['sup_baseline_e20_l1', 'sup_wide_e20_l1', 'sup_deep_e20_l1']

fig, ax = plt.subplots(figsize=(7, 4))
for run in arch_runs:
    h = load_history('supervised', run)
    ax.plot(h['train_loss'], label=f'{run} train')
    ax.plot(h['val_loss'], '--', label=f'{run} val')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig('../results/figures/arch_loss_curves.png', dpi=150)

In [ ]:
rows = []
for run in arch_runs:
    e = load_eval(run, 'test/Distance10mm')
    rows.append({'model': run, 'psnr': e['psnr']['mean'], 'ssim': e['ssim']['mean']})
pd.DataFrame(rows)

## 1-2: Epoch comparison (deep, 10 / 20 / 50 epochs)

In [ ]:
epoch_runs = {10: 'sup_deep_e10_l1', 20: 'sup_deep_e20_l1', 50: 'sup_deep_e50_l1'}
rows = []
for n_epochs, run in epoch_runs.items():
    e = load_eval(run, 'test/Distance10mm')
    rows.append({'epochs': n_epochs, 'psnr': e['psnr']['mean'], 'ssim': e['ssim']['mean']})
pd.DataFrame(rows)

## 1-3: Loss function comparison (L1 / L2 / L1+SSIM)

In [ ]:
loss_runs = {'L1': 'sup_deep_e20_l1', 'L2': 'sup_deep_e20_l2', 'L1+SSIM': 'sup_deep_e20_l1ssim'}
rows = []
for name, run in loss_runs.items():
    e = load_eval(run, 'test/Distance10mm')
    rows.append({'loss': name, 'psnr': e['psnr']['mean'], 'ssim': e['ssim']['mean']})
pd.DataFrame(rows)

## 1-4: Generalization across distances (5 / 10 / 15 / 20mm)

In [ ]:
distances = [5, 10, 15, 20]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, run in loss_runs.items():
    psnrs, ssims = [], []
    for d in distances:
        e = load_eval(run, f'test/Distance{d}mm')
        psnrs.append(e['psnr']['mean']); ssims.append(e['ssim']['mean'])
    axes[0].plot(distances, psnrs, marker='o', label=name)
    axes[1].plot(distances, ssims, marker='o', label=name)
axes[0].set_title('PSNR vs distance'); axes[1].set_title('SSIM vs distance')
for ax in axes: ax.set_xlabel('distance (mm)'); ax.legend()
fig.tight_layout()
fig.savefig('../results/figures/generalization.png', dpi=150)

## 2-1: Self-supervised, cross-distance evaluation

In [ ]:
ss_runs = ['ss_5_5', 'ss_10_10', 'ss_15_15', 'ss_20_20', 'ss_5_20']
table = {}
for run in ss_runs:
    row = {}
    for d in distances:
        e = load_eval(run, f'test/Distance{d}mm')
        row[f'{d}mm PSNR'] = e['psnr']['mean']
        row[f'{d}mm SSIM'] = e['ssim']['mean']
    table[run] = row
pd.DataFrame(table).T

## 2-2: Distance estimation (auto-focus)

In [ ]:
with open(RESULTS / 'depth_regressor' / 'depth_5_20' / 'history.json') as f:
    depth_hist = json.load(f)
pd.DataFrame(depth_hist['eval']).T

## Extra credit: OOD comparison (2mm / 25mm)

In [ ]:
with open(RESULTS / 'tables' / 'ood_comparison.json') as f:
    ood = json.load(f)
pd.DataFrame({k: {'psnr': v['psnr']['mean'], 'ssim': v['ssim']['mean']} for k, v in ood.items()}).T